In [1]:
import sympy as sp
import numpy as np
import sys

def run_bisection_posteriori():
    print("=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP CHIA ĐÔI ===")
    print("(Đánh giá điều kiện dừng bằng Sai số Hậu nghiệm)\n")
    
    try:
        # --- 0. Nhập liệu ---
        raw_f_str = input("Nhập hàm f(x) (ví dụ: x**3 - x - 1): ")
        f_str = raw_f_str.replace('^', '**') 
        
        a = float(input("Nhập đầu mút a: "))
        b = float(input("Nhập đầu mút b: "))
        
        if a > b:
            a, b = b, a
            
        epsilon = float(input("Nhập sai số mục tiêu epsilon (ví dụ: 1e-4): "))
        precision = int(input("Nhập số chữ số thập phân hiển thị (ví dụ: 5): "))

        # --- 1. Xử lý toán học ---
        x = sp.Symbol('x')
        f_expr = sp.sympify(f_str)
        f = sp.lambdify(x, f_expr, 'numpy')

        # --- 2. Kiểm tra điều kiện ---
        fa, fb = f(a), f(b)
        
        if np.isclose(fa, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm chính là tại mút a = {a}")
            return
        if np.isclose(fb, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm chính là tại mút b = {b}")
            return

        print("\n" + "="*60)
        print("### 1. Kiểm tra điều kiện an toàn")
        print(f"* **Hàm số:** $f(x) = {sp.latex(f_expr)}$")
        print(f"* **Khoảng phân li:** $f({a}) = {fa:.{precision}f}$, $f({b}) = {fb:.{precision}f}$")
        
        if fa * fb > 0:
            print(f"\n[!] LỖI: $f({a}) \\cdot f({b}) > 0$. Hàm không đổi dấu ở hai đầu mút. Không đảm bảo có nghiệm.")
            return
            
        print(f"=> $f({a}) \\cdot f({b}) < 0$. Đảm bảo tồn tại ít nhất 1 nghiệm trên $[{a}, {b}]$.")

        print("\n### 2. Xây dựng công thức và Đánh giá sai số")
        print("* **Công thức lặp:**")
        print("  $$c_n = \\frac{a_{n-1} + b_{n-1}}{2}$$")
        print("* **Công thức sai số hậu nghiệm:** (Nửa chiều dài khoảng hiện tại)")
        print(f"  $$\\Delta_n = \\frac{{b_{{n-1}} - a_{{n-1}}}}{{2}} \\le {epsilon:.1e}$$")

        # --- 3. Quá trình lặp (Sử dụng While - Hậu nghiệm) ---
        history = []
        curr_a, curr_b = a, b
        n = 1

        while True:
            c = (curr_a + curr_b) / 2
            fc = f(c)
            
            # Sai số hậu nghiệm tại bước hiện tại
            err = (curr_b - curr_a) / 2
            
            step_data = {
                'n': n, 'a': curr_a, 'b': curr_b, 'c': c, 
                'f(a)': f(curr_a), 'f(c)': fc, 'err': err, 'action': ''
            }
            
            # Dừng nếu vô tình rơi trúng nghiệm đúng xác thực
            if np.isclose(fc, 0, atol=1e-15):
                step_data['action'] = "Nghiệm đúng"
                history.append(step_data)
                break
                
            # Đánh giá cập nhật khoảng
            if f(curr_a) * fc < 0:
                step_data['action'] = "b_n = c"
                curr_b = c
            else:
                step_data['action'] = "a_n = c"
                curr_a = c
                
            history.append(step_data)

            # ĐIỀU KIỆN DỪNG HẬU NGHIỆM: Ngay khi sai số <= epsilon
            if err <= epsilon:
                break
                
            # Cầu chì an toàn chống lặp vô hạn
            if n > 1000:
                print("\n[!] CẢNH BÁO: Thuật toán không hội tụ sau 1000 bước.")
                break
                
            n += 1

        total_steps = len(history)

        # --- 4. Hiển thị chi tiết các bước ---
        print("\n### 3. Chi tiết các bước lặp tiêu biểu\n")
        
        # 2 BƯỚC ĐẦU
        print("**Hai bước lặp đầu tiên:**")
        for i in range(min(2, total_steps)):
            step = history[i]
            print(f"* **Bước $n = {step['n']}$:** Khoảng $[{step['a']:.{precision}f}, {step['b']:.{precision}f}]$")
            print(f"  $$c_{{{step['n']}}} = \\frac{{{step['a']:.{precision}f} + {step['b']:.{precision}f}}}{{2}} = {step['c']:.{precision}f}$$")
            print(f"  $$\\Delta_{{{step['n']}}} = \\frac{{{step['b']:.{precision}f} - {step['a']:.{precision}f}}}{{2}} = {step['err']:.{precision}g}$$")
            print(f"  * Lựa chọn: $f(c) {('< 0' if step['f(c)'] < 0 else '> 0')} \\Rightarrow$ Cập nhật {step['action']}")

        # 2 BƯỚC CUỐI
        if total_steps > 2:
            print("\n**Hai bước lặp cuối cùng:**")
            for i in range(max(2, total_steps - 2), total_steps):
                step = history[i]
                print(f"* **Bước $n = {step['n']}$:** Khoảng $[{step['a']:.{precision}f}, {step['b']:.{precision}f}]$")
                print(f"  $$c_{{{step['n']}}} = \\frac{{{step['a']:.{precision}f} + {step['b']:.{precision}f}}}{{2}} = {step['c']:.{precision}f}$$")
                print(f"  $$\\Delta_{{{step['n']}}} = \\frac{{{step['b']:.{precision}f} - {step['a']:.{precision}f}}}{{2}} = {step['err']:.{precision}g}$$")
                print(f"  * Lựa chọn: $f(c) {('< 0' if step['f(c)'] < 0 else '> 0')} \\Rightarrow$ Cập nhật {step['action']}")

        # --- 5. Bảng lặp rút gọn ---
        print("\n### 4. Bảng lặp tổng hợp rút gọn\n")
        print(f"| $n$ | $a_{{n-1}}$ | $b_{{n-1}}$ | $c_n$ | $f(c_n)$ | $\\Delta_n$ | Cập nhật |")
        print("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |")
        
        for i in range(min(3, total_steps)):
            s = history[i]
            print(f"| {s['n']} | {s['a']:.{precision}f} | {s['b']:.{precision}f} | {s['c']:.{precision}f} | {s['f(c)']:.{precision}f} | {s['err']:.{precision}g} | {s['action']} |")
            
        if total_steps > 6:
            print("| ... | ... | ... | ... | ... | ... | ... |")
            
        if total_steps > 3:
            for i in range(max(3, total_steps - 3), total_steps):
                s = history[i]
                print(f"| {s['n']} | {s['a']:.{precision}f} | {s['b']:.{precision}f} | {s['c']:.{precision}f} | {s['f(c)']:.{precision}f} | {s['err']:.{precision}g} | {s['action']} |")

        final_c = history[-1]['c']
        final_err = history[-1]['err']
        print(f"\n=> **KẾT LUẬN:** Qua {total_steps} bước lặp, nghiệm xấp xỉ là **$x^* \\approx {final_c:.{precision}f}$** (Sai số mắc phải $\\Delta = {final_err:.{precision}g} \\le {epsilon:.1e}$).")

    except Exception as e:
        print(f"\n[!] CÓ LỖI XẢY RA: {e}")

if __name__ == "__main__":
    run_bisection_posteriori()
    input("\nNhấn Enter để thoát chương trình...")

=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP CHIA ĐÔI ===
(Đánh giá điều kiện dừng bằng Sai số Hậu nghiệm)


### 1. Kiểm tra điều kiện an toàn
* **Hàm số:** $f(x) = x^{5} - 2401 x - 16807$
* **Khoảng phân li:** $f(7.0) = -16807.0000000$, $f(14.0) = 487403.0000000$
=> $f(7.0) \cdot f(14.0) < 0$. Đảm bảo tồn tại ít nhất 1 nghiệm trên $[7.0, 14.0]$.

### 2. Xây dựng công thức và Đánh giá sai số
* **Công thức lặp:**
  $$c_n = \frac{a_{n-1} + b_{n-1}}{2}$$
* **Công thức sai số hậu nghiệm:** (Nửa chiều dài khoảng hiện tại)
  $$\Delta_n = \frac{b_{n-1} - a_{n-1}}{2} \le 7.0e-06$$

### 3. Chi tiết các bước lặp tiêu biểu

**Hai bước lặp đầu tiên:**
* **Bước $n = 1$:** Khoảng $[7.0000000, 14.0000000]$
  $$c_{1} = \frac{7.0000000 + 14.0000000}{2} = 10.5000000$$
  $$\Delta_{1} = \frac{14.0000000 - 7.0000000}{2} = 3.5$$
  * Lựa chọn: $f(c) > 0 \Rightarrow$ Cập nhật b_n = c
* **Bước $n = 2$:** Khoảng $[7.0000000, 10.5000000]$
  $$c_{2} = \frac{7.0000000 + 10.5000000}{2} = 8.7500000$$
  $$\Del